In [45]:
from langgraph.graph import StateGraph,START,END
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel,Field
from typing import Annotated,Optional
from dotenv import load_dotenv
import operator
load_dotenv()

True

In [46]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto",  # let Hugging Face choose the best provider for you
)

chat_model = ChatHuggingFace(llm=llm)

In [47]:
class FeedBack(BaseModel):
    feedback: str = Field(description="Detailed feedback of the essay")
    score:int= Field(description='Score out of 0 to 10',ge=0,le=10)

In [48]:
essay = """India in the Age of AI
As the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.

India's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for All) with a focus on inclusive growth, aiming to leverage AI in healthcare, agriculture, education, and smart mobility.

One of the most promising applications of AI in India lies in agriculture, where predictive analytics can guide farmers on optimal sowing times, weather forecasts, and pest control. In healthcare, AI-powered diagnostics can help address India’s doctor-patient ratio crisis, particularly in rural areas. Educational platforms are increasingly using AI to personalize learning paths, while smart governance tools are helping improve public service delivery and fraud detection.

However, the path to AI-led growth is riddled with challenges. Chief among them is the digital divide. While metropolitan cities may embrace AI-driven solutions, rural India continues to struggle with basic internet access and digital literacy. The risk of job displacement due to automation also looms large, especially for low-skilled workers. Without effective skilling and re-skilling programs, AI could exacerbate existing socio-economic inequalities.

Another pressing concern is data privacy and ethics. As AI systems rely heavily on vast datasets, ensuring that personal data is used transparently and responsibly becomes vital. India is still shaping its data protection laws, and in the absence of a strong regulatory framework, AI systems may risk misuse or bias.

To harness AI responsibly, India must adopt a multi-stakeholder approach involving the government, academia, industry, and civil society. Policies should promote open datasets, encourage responsible innovation, and ensure ethical AI practices. There is also a need for international collaboration, particularly with countries leading in AI research, to gain strategic advantage and ensure interoperability in global systems.

India’s demographic dividend, when paired with responsible AI adoption, can unlock massive economic growth, improve governance, and uplift marginalized communities. But this vision will only materialize if AI is seen not merely as a tool for automation, but as an enabler of human-centered development.

In conclusion, India in the age of AI is a story in the making — one of opportunity, responsibility, and transformation. The decisions we make today will not just determine India’s AI trajectory, but also its future as an inclusive, equitable, and innovation-driven society."""

In [49]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
parser = PydanticOutputParser(pydantic_object=FeedBack)

# 4. Chain it together
# Using ChatHuggingFace as defined in your previous steps
# chain = prompt_template | chat_model | parser

# result = chain.invoke({"essay": essay}).score
# print(result)


In [50]:
class UPSCState(BaseModel):
    essay: str
    
    language_feedback: Optional[str] = None
    analysis_feedback: Optional[str] = None
    clarity_feedback: Optional[str] = None
    overall_feedback: Optional[str] = None
    
    individual_scores: Annotated[list[int], operator.add] = []
    
    avg_score: Optional[float] = None


In [51]:
def evaluate_language(state: UPSCState):
    prompt_template = PromptTemplate(
    template="Evaluate the language quality of the following essay.\n{format_instructions}\nEssay: {essay}",
    input_variables=["essay"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)
    chain = prompt_template|chat_model|parser
    result = chain.invoke({'essay':state.essay})
    return {'language_feedback':result.feedback,'individual_scores':[result.score]}

In [52]:
def evaluate_clarity(state: UPSCState):
    prompt_template = PromptTemplate(
    template="Evaluate the clarity in the language of the following essay.\n{format_instructions}\nEssay: {essay}",
    input_variables=["essay"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)
    chain = prompt_template|chat_model|parser
    result = chain.invoke({'essay':state.essay})
    return {'clarity_feedback':result.feedback,'individual_scores':[result.score]}

In [53]:

def evaluate_analysis(state: UPSCState):
    prompt_template = PromptTemplate(
    template="Evaluate the analze the quality of the following essay.\n{format_instructions}\nEssay: {essay}",
    input_variables=["essay"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)
    chain = prompt_template|chat_model|parser
    result = chain.invoke({'essay':state.essay})
    return {'analysis_feedback':result.feedback,'individual_scores':[result.score]}

In [61]:
def evaluate_feedback(state: UPSCState):
    prompt_template = PromptTemplate(
        template="Based on the following feedbacks create a summarized feedback.\n{format_instructions}\n"
                 "language feedback - {language_feedback}\n"
                 "depth of analysis feedback - {analysis_feedback}\n"
                 "clarity of thought feedback - {clarity_feedback}",
        input_variables=["language_feedback", "analysis_feedback", "clarity_feedback"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
    )
    chain = prompt_template | chat_model | parser
    result = chain.invoke({
        'language_feedback': state.language_feedback,
        'analysis_feedback': state.analysis_feedback,
        'clarity_feedback': state.clarity_feedback
    })
    return {'overall_feedback': result.feedback, 'avg_score': result.score}

In [62]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_clarity',evaluate_clarity)
graph.add_node('evaluate_feedback',evaluate_feedback)

graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_clarity')

graph.add_edge('evaluate_language','evaluate_feedback')
graph.add_edge('evaluate_analysis','evaluate_feedback')
graph.add_edge('evaluate_clarity','evaluate_feedback')

graph.add_edge('evaluate_feedback',END)

workflow = graph.compile()


In [63]:
initial_state = {
    'essay':essay
}

workflow.invoke(initial_state)

{'essay': "India in the Age of AI\nAs the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.\n\nIndia's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for 